<a href="https://colab.research.google.com/github/poojithabalusu/Garage-management-system/blob/main/job_recommendation_UI3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install gradio pandas scikit-learn


In [10]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr


In [11]:
file_path = "/content/drive/MyDrive/Assignment_ML/updated_job_dataset_with_education.csv"


In [12]:
df = pd.read_csv(file_path)

# Combine text
df['combined_text'] = (
    df['job_title'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['skills'].fillna('')
).str.lower()

# Vectorize job data
vectorizer = TfidfVectorizer(stop_words='english', max_features=8000)
job_vectors = vectorizer.fit_transform(df['combined_text'])

In [13]:
def recommend_jobs(skills, education, experience, top_n):
    user_profile = f"{skills} {education} {experience}".lower()
    user_vector = vectorizer.transform([user_profile])
    cos_sim = cosine_similarity(user_vector, job_vectors).flatten()
    top_indices = np.argsort(cos_sim)[::-1]

    filtered = []
    for idx in top_indices:
        if cos_sim[idx] < 0.01:
            continue
        required_edu = str(df.iloc[idx].get('education', '')).lower().strip()
        if required_edu in education.lower() or education.lower() in required_edu:
            filtered.append(idx)
        elif not required_edu or any(part in required_edu for part in education.lower().split()):
            filtered.append(idx)
        if len(filtered) >= top_n:
            break

    if not filtered:
        return "🚫 No matching jobs found."

    result = "## 🎯 Top Job Recommendations:\n\n"
    for idx in filtered:
        row = df.iloc[idx]
        result += f"### {row['job_title']} ({cos_sim[idx]*100:.2f}%)\n"
        result += f"- **Required Education**: {row.get('education', 'N/A')}\n"
        result += f"- **Skills**: {row['skills']}\n"
        result += f"- **Description**: {row['description'][:150]}...\n"
        result += f"- **Category**: `{row['category']}`\n"
        result += "---\n"
    return result


In [14]:
gr.Interface(
    fn=recommend_jobs,
    inputs=[
        gr.Textbox(label="Skills (comma separated)"),
        gr.Textbox(label="Education (e.g., B.Tech CSE, MBA HR, etc.)"),
        gr.Textbox(label="Experience (optional)"),
        gr.Slider(1, 10, value=5, label="Number of Recommendations")
    ],
    outputs="markdown",
    title="🎯 Smart Job Recommendation System",
    description="Get job matches based on your skills, education, and experience"
).launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eaa759ee6e1567a5ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
